# M0 · physical — grasp poses

Verify the simulated grasps on hardware and find the closure at which the hand *actually* grips
a brick. Input `../poses.json` (from [`../simulation/grasp_poses.ipynb`](../simulation/grasp_poses.ipynb)),
output `../poses_measured.json`.

**Stack:** `bc_stark_sdk` over RS-485 — see [`real_hand.py`](real_hand.py). The sim has no
contact model, so everything about *whether a grasp holds* has to be measured here.

The measurement that matters: motor **current**. A finger moving freely draws little; a finger
pressed against a brick draws a lot while its position stops changing. That pair — position
stalls, current climbs — is contact detection, and it is what M1 will use to know a brick is in
the hand.

> Keep a brick in your other hand and place it in the fingers when a cell asks. Nothing here
> moves the arm.

In [ ]:
import sys
from pathlib import Path

# Locate src/m0 whether the kernel started in this folder, in the repo root, or anywhere between.
# Matched by content, not by folder name, so it survives m0/M0 casing differences between
# a case-insensitive mac and the case-sensitive robot PC.
_here = [Path.cwd(), *Path.cwd().parents]
_candidates = [*_here, *(p / "src" / name for p in _here for name in ("m0", "M0"))]
M0_DIR = next((p for p in _candidates if (p / "hand_model.py").exists()), None)
if M0_DIR is None:
    raise RuntimeError(f"could not find src/m0 from {Path.cwd()}; open this notebook from its own folder")
for _path in (M0_DIR, M0_DIR / "physical"):
    if str(_path) not in sys.path:
        sys.path.insert(0, str(_path))

import time

import numpy as np

from hand_model import MEASURED_LIBRARY_PATH, POSE_LIBRARY_PATH, describe, load_poses, save_pose
from real_hand import RIGHT_HAND_ID, RealHand, RealHandUI

hand = RealHand(port=None, slave_id=RIGHT_HAND_ID, speed=300)
hand.connect()

ui = RealHandUI(hand)
ui

## 1. Replay the simulated library

With the hand **empty**. This measures the hand against itself: how closely it reaches a
free-air command, and which fingers cost current even unloaded.

In [ ]:
assert hand.armed, "press ARM in the panel above first"

poses = load_poses(POSE_LIBRARY_PATH)
report = {}

print(f"{'pose':22s} {'max |err|':>9s} {'max mA':>7s}")
for name, pose in sorted(poses.items()):
    result = hand.hold_and_measure(pose, duration=1.2, settle=0.6)
    report[name] = result
    currents = result["currents_mA"]
    print(f"{name:22s} {np.abs(result['error']).max():9.2f} {currents.max() if len(currents) else 0:7.0f}")
    hand.open_hand(duration=0.6)

hand.open_hand()

## 2. Close on a real brick

Put the brick between the thumb and index pads, then run the cell. It closes in 2 % steps and
stops as soon as either finger exceeds `current_mA`, so it finds the contact closure instead of
crushing the part.

Tune `current_mA` against the unloaded numbers from step 1: it needs to be above the free-air
draw and below what you are willing to press with. 250 mA is a starting point, not a
calibration.

In [ ]:
BRICK = "2x4 brick"
CONTACT_CURRENT_MA = 250.0

hand.open_hand()
input(f"place the {BRICK} between the thumb and index pads, then press Enter...")

result = hand.close_until_contact(
    fingers=("thumb", "index"),
    base_pose={"thumb_aux": 0.8},
    current_mA=CONTACT_CURRENT_MA,
    step=0.02,
    max_closure=0.85,
)

print(f"contact: {result['contact']}  closure: {result['closure']:.2f}  peak: {result['peak_mA']:.0f} mA\n")
print(f"{'closure':>8s} {'peak mA':>8s}")
for row in result["trace"]:
    print(f"{row['closure']:8.2f} {row['peak_mA']:8.0f}")

In [ ]:
# Does it hold? Lift the brick free with your fingers - it should resist.
print(describe(hand.target))
time.sleep(3.0)
print("measured:", hand.read_pose().round(2))
print("currents:", hand.currents().round(0))

In [ ]:
# Keep what the hand actually reached, not what we asked for.
if result["contact"]:
    name = "grip_" + BRICK.replace(" ", "_")
    save_pose(name, hand.read_pose(), MEASURED_LIBRARY_PATH)
    print(f"saved {name} -> {MEASURED_LIBRARY_PATH.name}: {hand.read_pose().round(2)}")
else:
    print("no contact detected - lower current_mA or raise max_closure")

hand.open_hand()

## 3. Repeatability

M1 will command the same grasp hundreds of times. Run the contact search a handful of times on
the same brick and look at the spread: that is the tolerance the grasp planner has to live
inside. A spread of more than a few percent means the pose is on a steep part of the aperture
curve — go back to the simulation notebook and pick a flatter one.

In [ ]:
REPEATS = 5

closures = []
for trial in range(REPEATS):
    hand.open_hand(duration=0.8)
    time.sleep(0.4)
    trial_result = hand.close_until_contact(
        fingers=("thumb", "index"), base_pose={"thumb_aux": 0.8},
        current_mA=CONTACT_CURRENT_MA, step=0.02, max_closure=0.85,
    )
    closures.append(trial_result["closure"] if trial_result["contact"] else np.nan)
    print(f"trial {trial + 1}: closure {closures[-1]:.2f}  peak {trial_result['peak_mA']:.0f} mA")

closures = np.array(closures, dtype=float)
print(f"\nmean {np.nanmean(closures):.3f}  std {np.nanstd(closures):.3f}  "
      f"range {np.nanmin(closures):.2f}..{np.nanmax(closures):.2f}")

hand.open_hand()

## 4. Sim vs hardware

The delta below is the model error you carry into M1: how far the geometric grasp from Drake is
from the one that actually holds a brick.

In [ ]:
designed = load_poses(POSE_LIBRARY_PATH)
measured = load_poses(MEASURED_LIBRARY_PATH, include_presets=False)

print(f"{'pose':22s} {'thumb':>7s} {'index':>7s}   (measured - designed)")
for name, pose in sorted(measured.items()):
    reference = designed.get(name.replace("grip_", "grasp_"))
    if reference is None:
        print(f"{name:22s} {'':>7s} {'':>7s}   no simulated counterpart")
        continue
    delta = pose - reference
    print(f"{name:22s} {delta[0]:7.2f} {delta[2]:7.2f}")

## Shutdown

In [ ]:
hand.close()

## Done when

- [ ] Every library pose replays on hardware and you know its free-air position error.
- [ ] The contact closure for a 2×4 brick is measured, saved, and repeatable to a few percent.
- [ ] You know the current that means "holding" versus "moving freely".

That last number is the grasp-success signal for M1/submodule 0: mount the hand on the RM75,
approach a brick with the TCP from [`../../scene.py`](../../scene.py), replay the saved grip,
and confirm the hold with the current reading before lifting.